# SAMueL find benchmark effect on each hospital

## Plain English summary
For each hospital, find the predicted thrombolysis use on their patients at each of the 25 benchmark hospitals.

## Load imports

In [1]:
import pandas as pd

from dataclasses import dataclass
from sklearn.model_selection import train_test_split

# Turn warnings off to keep notebook tidy
import warnings
warnings.filterwarnings("ignore")

## Set up paths and filenames

In [2]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    data_read_path: str = './data/'
    data_read_filename: str = 'reformatted_data_thrombolysis_decision.csv'
    data_save_path: str = './data'
    notebook: str = ''

paths = Paths()

# Load data



In [3]:
filename = paths.data_read_path + paths.data_read_filename
data = pd.read_csv(filename)


Ensure all values are float and shuffle

In [4]:
data = data.sample(frac=1.0, random_state=42)

## Limit to 10 features and thrombolysis label

In [5]:
features_to_use = [
    'stroke_team_id',
    'stroke_severity',
    'prior_disability',
    'age',
    'infarction',
    'onset_to_arrival_time',
    'precise_onset_known',
    'onset_during_sleep',
    'arrival_to_scan_time',
    'afib_anticoagulant',
    'year',    
    'thrombolysis'
]

data = data[features_to_use]

## Limit to arrivals within 4 hours

In [6]:
mask = data['onset_to_arrival_time'] <= 240
data = data[mask]

## Create stratification based on hospital and thrombolysis use

In [7]:
strat = data['stroke_team_id'].map(str) + '-' + data['thrombolysis'].map(str)

## Create and save 10k test and train sets

In [8]:
# Split X and y
X = data.drop('thrombolysis', axis=1)
y = data['thrombolysis']

# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, stratify=strat, random_state=42)
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)

# # Save
# train.to_csv(f'{paths.data_save_path}/cohort_10000_train.csv', index=False)
# test.to_csv(f'{paths.data_save_path}/cohort_10000_test.csv', index=False)